# 04 · From logical error rate to physical resources

**Goal:** connect this threshold data to FTQC resource estimation — the layer the FaultTolerantQChem project works at.

> **Honest scope:** this notebook uses a *standard fitted overhead model*, not data wired directly from FaultTolerantQChem. It shows the conceptual bridge between the two projects. Wiring the actual QPE/double-factorization logical counts in is a separate, real next step.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
sys.path.insert(0, os.path.abspath('.'))
import numpy as np

## The sub-threshold scaling law
Below threshold the logical error per logical qubit per round follows roughly

$$ p_L \approx A\,(p/p_{th})^{\lfloor (d+1)/2 \rfloor} $$

We use representative constants here (A ≈ 0.03, p_th ≈ 0.006). In a rigorous study you would *fit* A and p_th to the notebook-03 sweep data.

In [2]:
def logical_error_estimate(d, p, A=0.03, p_th=0.006):
    return A * (p / p_th) ** ((d + 1) // 2)

def distance_for_target(p, target_pL, A=0.03, p_th=0.006, dmax=51):
    for d in range(3, dmax, 2):           # odd distances only
        if logical_error_estimate(d, p, A, p_th) <= target_pL:
            return d
    return None

## Worked example
Suppose an algorithm needs a logical qubit to survive with error budget 1e-10 per round on hardware running at physical error rate p = 1e-3.

In [3]:
p_phys = 1e-3
target = 1e-10
d = distance_for_target(p_phys, target)
phys_per_logical = 2 * d ** 2          # ~2d^2 physical qubits per logical
print(f'required code distance : {d}')
print(f'physical qubits/logical: {phys_per_logical}')

required code distance : 21
physical qubits/logical: 882


## Putting it together with an algorithm's logical footprint
If FaultTolerantQChem reports, say, `n_logical` logical qubits and a logical depth `L` for a molecule's QPE circuit, the physical cost is:

In [4]:
n_logical = 100        # placeholder: from FaultTolerantQChem output
logical_depth = 1e6    # placeholder: algorithm logical cycles
cycle_time = 1e-6      # surface-code cycle time, seconds (hardware-dependent)

total_physical = n_logical * phys_per_logical
runtime_seconds = cycle_time * d * logical_depth
print(f'total physical qubits : {total_physical:,}')
print(f'estimated runtime     : {runtime_seconds:,.0f} s '
      f'({runtime_seconds/3600:.1f} h)')

total physical qubits : 88,200
estimated runtime     : 21 s (0.0 h)


## The one-sentence version for an interview
*Notebook 03 measures the logical error rate per distance; this notebook inverts that to pick the distance an algorithm needs, then multiplies by the per-logical-qubit overhead and the algorithm's logical footprint to get physical qubits and runtime. That footprint is exactly what FaultTolerantQChem produces.*

**What I would NOT claim:** that the constants here are fitted to my own data (they are representative), or that the two repos are wired into one pipeline (they are not — yet).